# 07 — 连续时序隐变量：卡尔曼滤波与观测 MLE
将第 03 章的线性连续隐变量增加时间依赖。仍是二维观测、标量隐变量。模型假设改变后，重新生成符合时序结构的数据。

$x_t=ax_{t-1}+w_t,\; w_t\sim N(0,Q)$；$y_t=h x_t+b+v_t,\; v_t\sim N(0,R)$。其中 $h$ 为二维向量，$R$ 为二维协方差矩阵，噪声独立，初始状态 $x_0\sim N(0,1)$。仅 $a$ 未知，其余参数固定。

目标：在不读取真实状态的情况下估计 $a$。滤波递推时，观测预测分布为 $N(hm_t^-+b,\; hP_t^-h^T+R)$。

In [ ]:
import numpy as np
from scipy.stats import multivariate_normal
from scipy.optimize import minimize_scalar
import matplotlib.pyplot as plt
rng=np.random.default_rng(42)
T=500; a_true=.85; Q=.2; R=.45**2*np.eye(2)
h=np.array([1.,.8]); b=np.array([0.,1.])
x=np.empty(T); x[0]=rng.normal()
for t in range(1,T): x[t]=a_true*x[t-1]+np.sqrt(Q)*rng.normal()
y=x[:,None]*h+b+rng.multivariate_normal(np.zeros(2),R,size=T)
def kalman(y,a):
    m,P=0.,1.
    estimates=[]; variances=[]; ll=[]
    for t,obs in enumerate(y):
        if t>0: m,P=a*m,a*a*P+Q
        pred=h*m+b
        S=P*np.outer(h,h)+R
        ll.append(multivariate_normal.logpdf(obs,mean=pred,cov=S))
        K=np.linalg.solve(S,P*h)
        m=m+K@(obs-pred)
        P=P-K@h*P
        estimates.append(m); variances.append(P)
    return np.array(estimates),np.array(variances),np.sum(ll)
result=minimize_scalar(lambda a:-kalman(y,a)[2],bounds=(-.99,.99),method='bounded')
a_hat=result.x
m,P,ll=kalman(y,a_hat)
print('MLE a:',a_hat,'simulation truth:',a_true)
t=np.arange(100)
plt.plot(t,x[:100],label='truth'); plt.plot(t,m[:100],label='filter'); plt.fill_between(t,m[:100]-2*np.sqrt(P[:100]),m[:100]+2*np.sqrt(P[:100]),alpha=.2); plt.legend(); plt.show()

真值 x 仅用于绘图。参数学习目标只依赖 y。这个模型的高维边缘似然可通过低维高斯递推计算，不需要 EM 或变分推断。

In [ ]:
candidates=np.linspace(-.95,.98,50)
plt.plot(candidates,[kalman(y,a)[2] for a in candidates]); plt.axvline(a_hat,ls='--'); plt.xlabel('a'); plt.ylabel('sequence log likelihood'); plt.show()

如果将线性状态转移或线性观测映射换成非线性神经网络，高斯噪声仍可保留，但预测状态和后验一般不再是高斯。这时需要近似推断，或在低维情况下进行数值积分。概率世界模型可以从这里进一步扩展；本项目未实现完整 PlaNet。